In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, sys, subprocess, torch
assert torch.cuda.is_available(), "GPU is off. Runtime > Change runtime type > select T4 GPU."

!pip install -q timm einops rasterio scikit-image fastapi "uvicorn[standard]" python-multipart pillow requests

if not os.path.exists('/content/SwinIR'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/JingyunLiang/SwinIR.git', '/content/SwinIR'], check=True)
if '/content/SwinIR' not in sys.path:
    sys.path.append('/content/SwinIR')
print("SwinIR repo ready:", os.path.exists('/content/SwinIR/models/network_swinir.py'))

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ------------------------- CONFIG -------------------------
WORK         = '/content/drive/MyDrive/SIH26142'           # same Drive folder as the training notebook
BUNDLE_PATH  = f'{WORK}/model_bundle.pt'                   # weights + BAND_SCALE (preferred)
BEST_CKPT    = f'{WORK}/swinir_4ch_best.pth'               # fallback: state_dict only
CACHE_NPZ    = f'{WORK}/cache_pairs_1000.npz'              # fallback: used to recompute BAND_SCALE

JOBS_DIR     = '/content/jobs'                             # temp outputs (not on Drive, kept fast)
PORT         = 8000

API_KEY         = ''          # empty = open API. Since the URL is public, it's better to set a secret key for the website
ALLOWED_ORIGINS = ['*']       # production: ['https://your-website.com']

MAX_CROP        = 768         # crop side (LR px). 768 -> 3072x3072 output. Don't go above this on a T4
MIN_CROP        = 32
MAX_UPLOAD_MB   = 90          # Cloudflare tunnel enforces roughly a 100MB body limit
MAX_QUEUE       = 10
KEEP_JOBS       = 20
PREVIEW_MAX     = 2048        # max side for PNG previews (GeoTIFF is always full-res)

os.makedirs(JOBS_DIR, exist_ok=True)
for p in (BUNDLE_PATH, BEST_CKPT, CACHE_NPZ):
    print(("FOUND   " if os.path.exists(p) else "missing ") + p)

In [ ]:
import numpy as np, json, time, re, uuid, shutil, threading, queue, io, math, traceback
import rasterio
from rasterio.transform import Affine
from rasterio.windows import Window
from models.network_swinir import SwinIR
from skimage.transform import resize
from skimage.filters import sobel

device = 'cuda'
SCALE, LR_PATCH, IN_CH = 2, 48, 4      # must EXACTLY match the training config

model = SwinIR(upscale=SCALE, in_chans=IN_CH, img_size=LR_PATCH, window_size=8, img_range=1.,
               depths=[6,6,6,6], embed_dim=60, num_heads=[6,6,6,6], mlp_ratio=2,
               upsampler='pixelshuffledirect', resi_connection='1conv').to(device)

def band_scale_from_cache():
    """Same logic as the training notebook: 99.5th percentile of the train split (70%, seed 42) LR images, rounded up to the nearest 100."""
    from sklearn.model_selection import train_test_split
    raw_lr = np.load(CACHE_NPZ)['lr']
    tr, _ = train_test_split(np.arange(len(raw_lr)), test_size=0.30, random_state=42)
    sub = raw_lr[tr].transpose(1, 0, 2, 3).reshape(raw_lr.shape[1], -1)[:, ::50].astype(np.float32)
    return (np.ceil(np.percentile(sub, 99.5, axis=1) / 100) * 100).astype(np.float32)

if os.path.exists(BUNDLE_PATH):
    b = torch.load(BUNDLE_PATH, map_location=device, weights_only=False)     # trusted, our own file
    model.load_state_dict(b['model'])
    BAND_SCALE = np.asarray(b['band_scale'], dtype=np.float32)
    WEIGHTS_SRC = f"bundle: {BUNDLE_PATH}"
elif os.path.exists(BEST_CKPT) and os.path.exists(CACHE_NPZ):
    model.load_state_dict(torch.load(BEST_CKPT, map_location=device))
    BAND_SCALE = band_scale_from_cache()
    torch.save(dict(model=model.state_dict(), band_scale=BAND_SCALE, bands=(1,2,3,7), scale=SCALE, lr_patch=LR_PATCH),
               BUNDLE_PATH)                                                   # so the next load is faster
    WEIGHTS_SRC = f"fallback ckpt + recomputed BAND_SCALE (bundle saved to {BUNDLE_PATH})"
else:
    raise FileNotFoundError("Neither model_bundle.pt nor (swinir_4ch_best.pth + cache_pairs_1000.npz) was found. Check the WORK path.")

model.eval()
assert BAND_SCALE.shape == (IN_CH,), f"BAND_SCALE has the wrong shape: {BAND_SCALE.shape}"
with torch.no_grad():                                                         # warm-up + shape sanity
    _y = model(torch.zeros(1, IN_CH, LR_PATCH, LR_PATCH, device=device))
assert _y.shape == (1, IN_CH, LR_PATCH * SCALE, LR_PATCH * SCALE), _y.shape
print("Weights:", WEIGHTS_SRC)
print("BAND_SCALE (B,G,R,NIR):", BAND_SCALE, "| params:", sum(p.numel() for p in model.parameters()))

In [ ]:
@torch.no_grad()
def sr2x_tiled(net, img_hwc, patch=48, margin=4, batch=16):
    """(H,W,C) [0,1] -> (2H,2W,C). Overlapping tiles + reflect padding; margin is cropped to avoid seams."""
    net.eval(); s = 2
    h, w, c = img_hwc.shape
    core = patch - 2 * margin
    ny, nx = -(-h // core), -(-w // core)
    P = np.pad(img_hwc, ((margin, ny*core + margin - h), (margin, nx*core + margin - w), (0, 0)), mode='reflect')
    out = np.zeros((ny*core*s, nx*core*s, c), np.float32)
    coords = [(y*core, x*core) for y in range(ny) for x in range(nx)]
    for i in range(0, len(coords), batch):
        chunk = coords[i:i+batch]
        t = torch.from_numpy(np.stack([P[y:y+patch, x:x+patch] for y, x in chunk]).transpose(0,3,1,2)).float().to(device)
        sr = net(t).clamp(0, 1).cpu().numpy().transpose(0,2,3,1)
        for (y, x), p in zip(chunk, sr):
            out[y*s:(y+core)*s, x*s:(x+core)*s] = p[margin*s:(patch-margin)*s, margin*s:(patch-margin)*s]
    return out[:h*s, :w*s]

def tile_uncertainty(net, img):
    """Stage-1 (5m) TTA uncertainty: std across 4 flips. Stage-2 uncertainty is not computed."""
    preds = []
    for fy in (False, True):
        for fx in (False, True):
            x = img[::-1] if fy else img
            x = x[:, ::-1] if fx else x
            p = sr2x_tiled(net, np.ascontiguousarray(x))
            p = p[:, ::-1] if fx else p
            p = p[::-1] if fy else p
            preds.append(p)
    preds = np.stack(preds)
    return preds.mean(0), preds.std(0).mean(-1)

def ndvi_dn(img_hwc, eps=1e-6):
    d = img_hwc * BAND_SCALE                      # channel order B,G,R,NIR
    return (d[..., 3] - d[..., 2]) / (d[..., 3] + d[..., 2] + eps)

def bicubic_up(img, f):
    h, w, c = img.shape
    return np.clip(resize(img, (h*f, w*f, c), order=3, mode='edge', anti_aliasing=False), 0, 1)

def block_down(img, f):
    h, w, c = img.shape
    return img.reshape(h//f, f, w//f, f, c).mean((1, 3))

def rgb_disp(img, ref=None):
    """B,G,R -> R,G,B for display; percentile stretch computed from `ref` so panels stay comparable."""
    ref = img if ref is None else ref
    r = ref[..., [2,1,0]]; lo, hi = np.percentile(r, 2), np.percentile(r, 98)
    return np.clip((img[..., [2,1,0]] - lo) / (hi - lo + 1e-6), 0, 1)

# ---- colour map / PNG helpers (avoid pyplot, keeps the server thread-safe) ----
from matplotlib import colormaps
from PIL import Image

def cmap_rgb(arr, cmap, vmin, vmax):
    a = np.clip((arr - vmin) / (vmax - vmin + 1e-9), 0, 1)
    return (colormaps[cmap](a)[..., :3] * 255).astype(np.uint8)

def save_png(arr, path):
    """arr: float [0,1] RGB or uint8 RGB. Preview capped at PREVIEW_MAX on the long side."""
    if arr.dtype != np.uint8:
        arr = (np.clip(arr, 0, 1) * 255).astype(np.uint8)
    im = Image.fromarray(arr)
    if max(im.size) > PREVIEW_MAX:
        im.thumbnail((PREVIEW_MAX, PREVIEW_MAX), Image.LANCZOS)
    im.save(path, format='PNG', compress_level=3)
    return im.size

print("Inference tools ready.")

In [ ]:
class InputError(ValueError):
    """Problem with the user's file/params (HTTP 400)."""

def parse_bands(band_order, bands_str):
    if bands_str and bands_str.strip():
        try:
            b = tuple(int(x) for x in bands_str.split(','))
        except ValueError:
            raise InputError("Invalid bands format. Example: 1,2,3,4")
        if len(b) != 4 or min(b) < 1:
            raise InputError("bands must contain exactly 4 positive band indexes (in B,G,R,NIR order)")
        return b
    presets = {'copernicus': (1, 2, 3, 4), 'sen2venus': (1, 2, 3, 7)}
    if band_order not in presets:
        raise InputError(f"band_order '{band_order}' unknown. Options: {list(presets)} or bands='1,2,3,4'")
    return presets[band_order]

def probe_tile(path, bands):
    try:
        with rasterio.open(path) as src:
            info = dict(width=src.width, height=src.height, band_count=src.count,
                        crs=str(src.crs) if src.crs else None,
                        pixel_size=[abs(src.res[0]), abs(src.res[1])], dtype=src.dtypes[0],
                        geographic=bool(src.crs and src.crs.is_geographic))
    except Exception as e:
        raise InputError(f"File could not be read as a GeoTIFF: {str(e)[:160]}")
    if info['band_count'] < max(bands):
        raise InputError(f"File has {info['band_count']} band(s), but band index {max(bands)} is required. "
                         f"Upload a 4-band GeoTIFF (B02,B03,B04,B08) or adjust the bands parameter.")
    if info['dtype'] == 'uint8':
        raise InputError("8-bit (uint8) true-colour/visual exports are not supported (they do not carry reflectance values). "
                         "Export the raw bands B02,B03,B04,B08 (uint16 or float reflectance) from Copernicus.")
    return info

def read_window(path, bands, x0, y0, size):
    with rasterio.open(path) as src:
        x0 = int(np.clip(x0, 0, max(src.width - 1, 0))); y0 = int(np.clip(y0, 0, max(src.height - 1, 0)))
        w = Window(x0, y0, min(size, src.width - x0), min(size, src.height - y0))
        arr = src.read(list(bands), window=w).astype(np.float32)
        crs, tf = src.crs, src.window_transform(w)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    if arr.max() <= 0:
        raise InputError("Selected window is empty/nodata (all values zero). Try different x0, y0.")
    scaled = False
    if arr.max() <= 5:                      # 0-1 reflectance export -> DN (x10000), matching the training data
        arr = arr * 10000.0; scaled = True
    img = np.clip(arr.transpose(1, 2, 0) / BAND_SCALE, 0, 1)
    return img, crs, tf, dict(window=[x0, y0, int(w.width), int(w.height)], reflectance_x10000=scaled)

def write_geotiff(path, arr_chw, crs, transform, descriptions=None):
    with rasterio.open(path, 'w', driver='GTiff', height=arr_chw.shape[1], width=arr_chw.shape[2],
                       count=arr_chw.shape[0], dtype='float32', crs=crs, transform=transform,
                       compress='deflate', predictor=3) as dst:
        dst.write(arr_chw.astype(np.float32))
        if descriptions:
            dst.descriptions = tuple(descriptions)

def check_georef(img, crs, tf, sr_path, f):
    with rasterio.open(sr_path) as sr:
        ok = (sr.crs == crs
              and abs(sr.transform.a - tf.a / f) <= 1e-6 * max(1.0, abs(tf.a))
              and abs(sr.transform.c - tf.c) <= 1e-6 * max(1.0, abs(tf.c))
              and abs(sr.transform.f - tf.f) <= 1e-6 * max(1.0, abs(tf.f))
              and sr.width == img.shape[1] * f and sr.height == img.shape[0] * f)
        return dict(passed=bool(ok), crs_in=str(crs) if crs else None, crs_out=str(sr.crs) if sr.crs else None,
                    pixel_size_in=abs(tf.a), pixel_size_out=abs(sr.transform.a),
                    origin_in=[tf.c, tf.f], origin_out=[sr.transform.c, sr.transform.f],
                    message="CRS same, pixel size / 4, origin same, footprint same" if ok else "Georeferencing MISMATCH")

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def process_tile(path, job_dir, bands=(1, 2, 3, 4), x0=0, y0=0, size=256, want_unc=True, report=lambda s, p: None):
    t_all = time.time()
    job_id = os.path.basename(job_dir)
    os.makedirs(job_dir, exist_ok=True)
    size = int(np.clip(size, MIN_CROP, MAX_CROP))
    warnings = []

    report('reading', 0.05)
    meta = probe_tile(path, bands)
    if meta['geographic']:
        warnings.append("CRS is geographic (degrees). Pixel size is not in metres; use a UTM-projected Sentinel-2 tile.")
    elif not (5 <= meta['pixel_size'][0] <= 30):
        warnings.append(f"Pixel size is {meta['pixel_size'][0]:.2f} m; the model is trained for 10 m Sentinel-2.")
    if meta['crs'] is None:
        warnings.append("File has no CRS; the output will not be georeferenced either.")
    img, crs, tf, wmeta = read_window(path, bands, x0, y0, size)
    h, w = img.shape[:2]
    if min(h, w) < MIN_CROP:
        raise InputError(f"Crop is too small ({w}x{h}). At least {MIN_CROP}x{MIN_CROP} px is required.")

    timing = {}
    t = time.time(); report('stage1_5m', 0.15)
    st1 = sr2x_tiled(model, img)                       # 10m -> 5m   (real-validated on SEN2VENUS)
    report('stage2_2.5m', 0.30)
    st2 = sr2x_tiled(model, st1)                       # 5m -> 2.5m  (cascade, extrapolated)
    timing['cascade_s'] = round(time.time() - t, 2)

    unc = None
    if want_unc:
        t = time.time(); report('uncertainty', 0.65)
        _, unc = tile_uncertainty(model, img)
        timing['uncertainty_s'] = round(time.time() - t, 2)

    report('writing_outputs', 0.88)
    bc2, bc4 = bicubic_up(img, 2), bicubic_up(img, 4)
    n_bc, n_sr = ndvi_dn(bc4), ndvi_dn(st2)
    e_bc, e_sr = sobel(bc4[..., :3].mean(-1)), sobel(st2[..., :3].mean(-1))
    tf4 = tf * Affine.scale(1 / 4, 1 / 4)

    # ---- GeoTIFF outputs (full-res, georeferenced) ----
    write_geotiff(f'{job_dir}/SR_2p5m.tif', (st2 * BAND_SCALE).transpose(2, 0, 1), crs, tf4,
                  ['B02_blue', 'B03_green', 'B04_red', 'B08_nir'])
    write_geotiff(f'{job_dir}/NDVI_2p5m.tif', n_sr[None].astype(np.float32), crs, tf4, ['NDVI'])
    geo = check_georef(img, crs, tf, f'{job_dir}/SR_2p5m.tif', 4)

    # ---- PNG previews ----
    ev = float(max(np.percentile(np.concatenate([e_bc.ravel(), e_sr.ravel()]), 99), 1e-9))
    pngs = {
        'input_10m':        rgb_disp(img, img),
        'bicubic_2p5m':     rgb_disp(bc4, bc4),
        'sr_5m':            rgb_disp(st1, bc2),
        'sr_2p5m':          rgb_disp(st2, bc4),
        'ndvi_bicubic':     cmap_rgb(n_bc, 'RdYlGn', -1, 1),
        'ndvi_sr':          cmap_rgb(n_sr, 'RdYlGn', -1, 1),
        'ndvi_diff':        cmap_rgb(n_sr - n_bc, 'coolwarm', -0.3, 0.3),
        'edges_bicubic':    cmap_rgb(e_bc, 'gray', 0, ev),
        'edges_sr':         cmap_rgb(e_sr, 'gray', 0, ev),
    }
    legends = {'ndvi': [-1, 1], 'ndvi_diff': [-0.3, 0.3], 'edges': [0, ev]}
    if unc is not None:
        umax = float(max(np.percentile(unc, 99), 1e-9))
        pngs['uncertainty'] = cmap_rgb(unc, 'inferno', 0, umax)
        legends['uncertainty'] = [0, umax]
    sizes = {k: list(save_png(v, f'{job_dir}/{k}.png')) for k, v in pngs.items()}

    metrics = dict(
        lr_consistency_rmse_cascade=rmse(block_down(st2, 4), img),
        lr_consistency_rmse_bicubic=rmse(block_down(bc4, 4), img),
        ndvi_mean_sr=float(n_sr.mean()), ndvi_mean_bicubic=float(n_bc.mean()),
        edge_strength_sr=float(e_sr.mean()), edge_strength_bicubic=float(e_bc.mean()),
        uncertainty_mean=float(unc.mean()) if unc is not None else None,
        uncertainty_p95=float(np.percentile(unc, 95)) if unc is not None else None,
    )
    timing['total_s'] = round(time.time() - t_all, 2)

    return dict(
        job_id=job_id,
        input=dict(**meta, **wmeta, bands_used=list(bands), lr_size=[w, h]),
        output=dict(size=[w * 4, h * 4], pixel_size=abs(tf4.a), scale_factor=4,
                    preview_sizes=sizes, band_order=['B02', 'B03', 'B04', 'B08']),
        images={k: f'/jobs/{job_id}/{k}.png' for k in pngs},
        downloads={'sr_geotiff': f'/download/{job_id}/SR_2p5m.tif', 'ndvi_geotiff': f'/download/{job_id}/NDVI_2p5m.tif'},
        legends=legends, georef=geo, metrics=metrics, timing=timing, warnings=warnings,
        notes=["Stage-1 (10m->5m) is validated against real VENUS ground truth from SEN2VENUS.",
               "Stage-2 (5m->2.5m) is a cascade extrapolation: no direct ground truth exists at this scale. Fine details are model-inferred, not observed.",
               "The uncertainty map is TTA disagreement (std) at stage-1 (5m); it is not a calibrated probability.",
               "Edge strength indicates feature visibility, not proof of accuracy."],
    )

print("process_tile ready.")

In [ ]:
def make_synthetic_tile(path, n=200):
    """Fake 4-band uint16, UTM 46N, 10 m. For pipeline/plumbing checks only, not a quality benchmark."""
    from rasterio.transform import from_origin
    yy, xx = np.mgrid[0:n, 0:n].astype(np.float32)
    base = 900 + 400 * np.sin(xx / 17) + 300 * np.cos(yy / 23)
    rng = np.random.default_rng(0)
    bands = np.stack([base * 0.6, base * 0.8, base * 0.9, base * 1.6]) + rng.normal(0, 25, (4, n, n))
    bands[:, n//2-1:n//2+1, :] = 2200                      # 'road'
    bands[:, 20:50, 20:60] = 2600                          # 'building block'
    with rasterio.open(path, 'w', driver='GTiff', height=n, width=n, count=4, dtype='uint16',
                       crs='EPSG:32646', transform=from_origin(390000, 2960000, 10, 10)) as dst:
        dst.write(np.clip(bands, 0, 65535).astype(np.uint16))
    return path

_t = make_synthetic_tile('/content/synthetic_tile.tif')
_res = process_tile(_t, f'{JOBS_DIR}/selftest', size=128, want_unc=True)
assert _res['georef']['passed'], _res['georef']
assert all(os.path.exists(f"{JOBS_DIR}/selftest/{k}.png") for k in _res['images'])
print("SELF-TEST PASSED | output size:", _res['output']['size'], "| timing:", _res['timing'])
print("metrics:", {k: (round(v, 5) if isinstance(v, float) else v) for k, v in _res['metrics'].items()})
shutil.rmtree(f'{JOBS_DIR}/selftest', ignore_errors=True)

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles

app = FastAPI(title="SIH26142 SRM Inference API", version="1.0")
app.add_middleware(CORSMiddleware, allow_origins=ALLOWED_ORIGINS, allow_credentials=False,
                   allow_methods=["*"], allow_headers=["*"])

JOBS, JOBS_LOCK, JOB_Q = {}, threading.Lock(), queue.Queue()
DOWNLOADABLE = {'SR_2p5m.tif', 'NDVI_2p5m.tif'}

def _set(job_id, **kw):
    with JOBS_LOCK:
        if job_id in JOBS: JOBS[job_id].update(kw)

def _queue_position(job_id):
    with JOBS_LOCK:
        ids = [k for _, k in sorted((j['created'], k) for k, j in JOBS.items() if j['status'] == 'queued')]
    return ids.index(job_id) + 1 if job_id in ids else 0

def _cleanup():
    with JOBS_LOCK:
        done = sorted((j['created'], k) for k, j in JOBS.items() if j['status'] in ('done', 'error'))
        drop = [k for _, k in done[:-KEEP_JOBS]] if len(done) > KEEP_JOBS else []
        for k in drop: JOBS.pop(k, None)
    for k in drop: shutil.rmtree(f'{JOBS_DIR}/{k}', ignore_errors=True)

def worker():
    """Single GPU -> one job at a time. The rest wait in the queue."""
    while True:
        job_id = JOB_Q.get()
        try:
            job = JOBS[job_id]
            _set(job_id, status='running', stage='starting', progress=0.02, started=time.time())
            res = process_tile(job['input_path'], f'{JOBS_DIR}/{job_id}', bands=job['bands'], x0=job['x0'], y0=job['y0'],
                               size=job['size'], want_unc=job['unc'],
                               report=lambda s, p, _j=job_id: _set(_j, stage=s, progress=float(p)))
            _set(job_id, status='done', stage='done', progress=1.0, result=res)
        except InputError as e:
            _set(job_id, status='error', error=str(e), error_code='bad_input')
        except torch.cuda.OutOfMemoryError:
            _set(job_id, status='error', error_code='gpu_oom',
                 error='Ran out of GPU memory. Reduce crop_size or turn off uncertainty.')
        except Exception as e:
            traceback.print_exc()
            _set(job_id, status='error', error_code='internal', error=f'Internal error: {str(e)[:200]}')
        finally:
            try: os.remove(JOBS[job_id]['input_path'])          # no need to keep the user's input file
            except Exception: pass
            torch.cuda.empty_cache(); _cleanup()

def _check_key(key):
    if API_KEY and key != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid or missing X-API-Key")

@app.get("/")
def root():
    return {"service": "SIH26142 SRM inference", "endpoints": ["/health", "POST /infer", "/status/{job_id}", "/download/{job_id}/{name}"]}

@app.get("/health")
def health():
    with JOBS_LOCK:
        counts = {s: sum(1 for j in JOBS.values() if j['status'] == s) for s in ('queued', 'running', 'done', 'error')}
    return dict(status="ok", gpu=torch.cuda.get_device_name(0), weights=WEIGHTS_SRC, scale_factor=4,
                band_scale=[float(x) for x in BAND_SCALE], max_crop=MAX_CROP, max_upload_mb=MAX_UPLOAD_MB,
                api_key_required=bool(API_KEY), jobs=counts)

@app.post("/infer", status_code=202)
def infer(file: UploadFile = File(...), band_order: str = Form("copernicus"), bands: str = Form(""),
          x0: int = Form(0), y0: int = Form(0), crop_size: int = Form(256), uncertainty: bool = Form(True),
          x_api_key: str = Header(default="")):
    _check_key(x_api_key)
    try:
        bands_t = parse_bands(band_order, bands)
    except InputError as e:
        raise HTTPException(status_code=400, detail=str(e))
    with JOBS_LOCK:
        waiting = sum(1 for j in JOBS.values() if j['status'] == 'queued')
    if waiting >= MAX_QUEUE:
        raise HTTPException(status_code=503, detail="Server busy (queue full). Try again shortly.")

    job_id = uuid.uuid4().hex
    job_dir = f'{JOBS_DIR}/{job_id}'; os.makedirs(job_dir, exist_ok=True)
    in_path, total = f'{job_dir}/input.tif', 0
    with open(in_path, 'wb') as f:
        while chunk := file.file.read(1 << 20):
            total += len(chunk)
            if total > MAX_UPLOAD_MB * 1e6:
                f.close(); shutil.rmtree(job_dir, ignore_errors=True)
                raise HTTPException(status_code=413, detail=f"File is larger than {MAX_UPLOAD_MB} MB. Upload a smaller crop/GeoTIFF.")
            f.write(chunk)
    try:
        probe_tile(in_path, bands_t)                       # quick validation, fails fast with 400
    except InputError as e:
        shutil.rmtree(job_dir, ignore_errors=True)
        raise HTTPException(status_code=400, detail=str(e))

    with JOBS_LOCK:
        JOBS[job_id] = dict(status='queued', stage='queued', progress=0.0, created=time.time(), input_path=in_path,
                            bands=bands_t, x0=x0, y0=y0, size=crop_size, unc=uncertainty,
                            filename=file.filename, result=None, error=None, error_code=None)
    JOB_Q.put(job_id)
    return dict(job_id=job_id, status='queued', queue_position=_queue_position(job_id), status_url=f'/status/{job_id}')

@app.get("/status/{job_id}")
def status(job_id: str):
    if not re.fullmatch(r'[0-9a-f]{32}', job_id) or job_id not in JOBS:
        raise HTTPException(status_code=404, detail="Unknown job_id (expired or the server restarted)")
    with JOBS_LOCK:
        j = dict(JOBS[job_id])
    out = dict(job_id=job_id, status=j['status'], stage=j['stage'], progress=round(j['progress'], 3),
               queue_position=_queue_position(job_id) if j['status'] == 'queued' else 0)
    if j['status'] == 'done': out['result'] = j['result']
    if j['status'] == 'error': out.update(error=j['error'], error_code=j['error_code'])
    return out

@app.get("/download/{job_id}/{name}")
def download(job_id: str, name: str):
    p = f'{JOBS_DIR}/{job_id}/{name}'
    if not re.fullmatch(r'[0-9a-f]{32}', job_id) or name not in DOWNLOADABLE or not os.path.exists(p):
        raise HTTPException(status_code=404, detail="File not found")
    return FileResponse(p, media_type='image/tiff', filename=name)

app.mount("/jobs", StaticFiles(directory=JOBS_DIR), name="jobs")      # PNG previews

if 'WORKER' not in globals():
    WORKER = threading.Thread(target=worker, daemon=True); WORKER.start()
print("API defined, worker thread running.")

In [ ]:
import uvicorn, requests

if 'server' in globals() and getattr(server, 'started', False):
    print("Server already running on port", PORT)
else:
    server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=PORT, log_level='warning',
                                           proxy_headers=True, forwarded_allow_ips='*'))
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(50):
        try:
            if requests.get(f'http://127.0.0.1:{PORT}/health', timeout=2).ok: break
        except Exception:
            time.sleep(0.5)
    print("Local server up:", requests.get(f'http://127.0.0.1:{PORT}/health').json())

In [ ]:
# ---------------- TUNNEL ----------------
# 'cloudflared': no signup needed, but the URL changes on EVERY run (the website's API_BASE must be updated each time).
# 'ngrok'      : free account + free static domain (dashboard.ngrok.com > Domains) = FIXED URL. Better for the live website.
TUNNEL       = 'cloudflared'
NGROK_TOKEN  = ''
NGROK_DOMAIN = ''            # e.g. 'your-name.ngrok-free.app'

import urllib.request, collections
public_url = None

if TUNNEL == 'ngrok':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], check=True)
    from pyngrok import ngrok
    assert NGROK_TOKEN, "Set NGROK_TOKEN"
    ngrok.set_auth_token(NGROK_TOKEN)
    try:
        tun = ngrok.connect(PORT, "http", domain=NGROK_DOMAIN) if NGROK_DOMAIN else ngrok.connect(PORT, "http")
    except TypeError:
        tun = ngrok.connect(PORT, "http")
    public_url = tun.public_url.replace('http://', 'https://')
else:
    exe = '/usr/local/bin/cloudflared'
    if not os.path.exists(exe):
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', exe)
        os.chmod(exe, 0o755)
    proc = subprocess.Popen([exe, 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail, found = collections.deque(maxlen=30), threading.Event()
    def _drain():
        global public_url
        for line in proc.stdout:                         # the pipe must be drained continuously
            tail.append(line.rstrip())
            m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
            if m and not public_url:
                public_url = m.group(0); found.set()
    threading.Thread(target=_drain, daemon=True).start()
    if not found.wait(45):
        print("Tunnel URL not found. cloudflared log:\n" + "\n".join(tail))

if public_url:
    open(f'{WORK}/backend_url.txt', 'w').write(public_url)
    time.sleep(3)
    try:
        r = requests.get(public_url + '/health', headers={'ngrok-skip-browser-warning': '1'}, timeout=20)
        print("Public health check:", r.status_code)
    except Exception as e:
        print("Public health check failed (give the tunnel a few more seconds, then retry):", e)
    print("\n" + "=" * 60 + f"\nAPI_BASE = {public_url}\n" + "=" * 60)
    print("(This URL was also saved to:", f'{WORK}/backend_url.txt)')

In [ ]:
def http_roundtrip(base, tif_path, **form):
    h = {'ngrok-skip-browser-warning': '1'}
    if API_KEY: h['X-API-Key'] = API_KEY
    with open(tif_path, 'rb') as f:
        r = requests.post(base + '/infer', files={'file': ('tile.tif', f)}, data=form, headers=h, timeout=60)
    r.raise_for_status(); job = r.json(); print("submitted:", job)
    while True:
        s = requests.get(base + job['status_url'], headers=h, timeout=30).json()
        print(f"  {s['status']:8s} {s['stage']:16s} {s['progress']*100:5.1f}%")
        if s['status'] in ('done', 'error'): return s, h
        time.sleep(1.5)

base = f'http://127.0.0.1:{PORT}'             # to test against the public URL instead, set base = public_url
s, h = http_roundtrip(base, '/content/synthetic_tile.tif', band_order='copernicus', crop_size=128, uncertainty='true')
assert s['status'] == 'done', s
res = s['result']
print("images:", list(res['images']))
for k in ('sr_2p5m', 'uncertainty'):
    r = requests.get(base + res['images'][k], headers=h); print(k, r.status_code, r.headers['content-type'], len(r.content), "bytes")
r = requests.get(base + res['downloads']['sr_geotiff'], headers=h); print("SR GeoTIFF download:", r.status_code, len(r.content), "bytes")
bad = requests.post(base + '/infer', files={'file': ('x.tif', b'not a tiff')}, headers=h, timeout=30)
print("bad file ->", bad.status_code, bad.json())
print("HTTP round-trip OK")

In [ ]:
print("Serving at:", public_url or f"http://127.0.0.1:{PORT} (no tunnel)")
try:
    while True:
        time.sleep(120)
        with JOBS_LOCK:
            n = {s: sum(1 for j in JOBS.values() if j['status'] == s) for s in ('queued', 'running', 'done', 'error')}
        print(time.strftime('%H:%M:%S'), "alive | jobs:", n, "|", public_url)
except KeyboardInterrupt:
    print("Stopped keep-alive (the server thread keeps running as long as the runtime is alive).")